### 正则化与优化（Regularization & Optimization）
#### Regularization
<div align="center">
  <img src="class_images/regularization.jpg" width="500">
</div>

* 引入正则项的作用：设置偏好/防止过拟合
* 上图中的$\lambda$是正则化强度（超参数）

<div align="center">
  <img src="class_images/full_loss.jpg" width="500">
</div>

#### Optimization
通过 Analytic gradient 计算梯度，确定参数$W$的修改方向
* 学习率：一些动态调整学习率的方法<br>
（1）在特定节点降低学习率，例如ResNets中，分别在第30，60，90次epoch后将学习率乘以0.1<br>
（2）Cosine：$\alpha_t = \frac{1}{2}\alpha_0(1+\cos\frac{t\pi}{T})$，其中$t$是第$t$个epoch，$T$是epoch的总数<br>
（3）线性：$\alpha_t = \alpha_0(1-\frac{t}{T})$<br>
（4）Inverse sqrt：$\alpha_t = \frac{\alpha_0}{\sqrt{t}}$<br>
（5）Linear Warmup<br>
**梯度下降（Gradient Descent）**：常用的梯度下降方法：SGD（随机梯度下降），其优势是每次选择一个小批次计算梯度，否则损失计算太贵;总体样本分为多个批次训练完一轮，称为一个epoch
* SGD的问题：<br>
损失沿不同方向变化的速度可能相差很大，一个太慢，另一个太快<br>
可能会卡在局部最小值<br>
因为每次训练只是取总体训练集的一部分进行梯度下降，因此训练过程中损失函数的变化方向不一定每次都沿着总体而言最优的方向<br>
解决办法：引入Momentum<br>

In [ ]:
# SGD
while True:
    data_batch = sample_training_data(data, batch_size)
    weight_grad = evaluate_gradient(loss_fun, data_batch, weights)
    weights += - learning_rate * weight_grad

In [ ]:
# SGD + Momentum
vx = 0
while True:
    dx = compute_gradient(x)
    vx = rho * vx + dx
    x -= learning_rate * vx

**另一种更复杂的优化：RMSProp**：它的好处是通过除以(np.sqrt(grad_squared) + eps)，使得损失函数在更陡的方向上变化更小，在更平坦的方向上变化更大

In [ ]:
# RMSProp
grad_squared = 0
while True:
    dx = compute_gradient(x)
    grad_squared = decay_rate * grad_squared + (1 - decay_rate) * dx**2
    x -= learning_rate * dx / (np.sqrt(grad_squared) + eps)

**当下深度学习最流行的优化器：Adam**：可以认为它结合了之前两种优化器，其中first_moment起到Momentum的作用，而second_moment起到RMSProp的作用；由于在初始步骤时，first_moment和second_moment为0，而beta1和beta2通常是非常接近1的数，因此初始变化会很大，于是引入first_unbias和second_unbias来解决这个问题
* beta1 = 0.9，beta2 = 0.999，learning_rate = 1e-3或5e-4对很多模型是一个好的初始设置

In [ ]:
# Adam
first_moment = 0
second_moment = 0
for t in range(1, num_iterations + 1):
    dx = compute_gradient(x)
    first_moment = beta1 * first_moment + (1 - beta1) * dx
    second_moment = beta2 * second_moment + (1 - beta2) * dx**2
    first_unbias = first_moment / (1 - beta1**t)
    second_unbias = second_moment / (1 - beta2**t)
    x -= learning_rate * first_unbias / (np.sqrt(second_unbias) + eps)